# Population Aerotaxis — temporal analysis

Behavioural state (speed, reversals, turns) locked to **global gas shifts**.
Works on the tidy table from `create_results_dict_server.py`
(`aerotaxis_results.{parquet,csv,pkl}`) with columns:
`Condition, Recording, Crop_ID, Frame, Time_Seconds, O2_State, Forward_Velocity, Reversal_Active, Turn_Active`.

Analysis primitives live in `toolscripts/utils/aerotaxis_analysis.py` so the same
code runs here, in a plain script, or feeds tidy CSVs to R.

### 1. Load the tidy results

In [ ]:
import sys
from pathlib import Path

# make the analysis helpers importable (edit if your layout differs)
UTILS = Path.cwd().parents[1] / 'toolscripts' / 'utils'
if UTILS.exists():
    sys.path.insert(0, str(UTILS))
import aerotaxis_analysis as aa

# point this at your combined results file OR the dataset folder
RESULTS = 'aerotaxis_results.parquet'
df = aa.load_results(RESULTS)
print(f'{len(df):,} rows | states: {sorted(df.O2_State.unique())}')
df.head()

### 2. Per-state summary
Reversal/turn columns are 0/1, so their means are the **fraction of time** in that state.

In [ ]:
summary = aa.per_state_summary(df)
summary

### 3. Behaviour by gas state (per condition)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, ['mean_forward_velocity', 'reversal_fraction', 'turn_fraction']):
    aa.plot_per_state_summary(summary, metric=m, ax=ax)
fig.tight_layout()

### 4. Gas-transition-triggered averages
Align a feature to every O2 shift (t=0 at the shift) and average across crops.
This is the core aerotaxis readout — the on/off dynamics of the O2 response.

In [ ]:
tta = aa.transition_triggered_average(df, feature='Forward_Velocity',
                                      pre_s=10, post_s=30, fps=10)
fig, ax = plt.subplots(figsize=(9, 4))
aa.plot_transition_triggered(tta, feature='Forward_Velocity', ax=ax)
plt.show()

# swap the feature for the reversal / turn response:
# tta_rev = aa.transition_triggered_average(df, feature='Reversal_Active', pre_s=10, post_s=30)

### 5. Per-crop time-series viewer
Interactive (ipywidgets): pick a crop, inspect features with the gas protocol shaded.

In [ ]:
import ipywidgets as widgets
import numpy as np

df['_crop'] = df[aa.GROUP_KEYS].agg(' / '.join, axis=1)
crops = sorted(df['_crop'].unique())
states = sorted(df.O2_State.unique())
cmap = {s: c for s, c in zip(states, plt.cm.tab10.colors)}

def show(crop):
    sub = df[df['_crop'] == crop].sort_values('Frame')
    t = sub.Time_Seconds.to_numpy()
    fig, ax = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(t, sub.Forward_Velocity); ax[0].set_ylabel('fwd vel\n(mm/s)')
    ax[1].plot(t, sub.Reversal_Active); ax[1].set_ylabel('reversal')
    ax[2].plot(t, sub.Turn_Active); ax[2].set_ylabel('turn'); ax[2].set_xlabel('time (s)')
    # shade gas states
    st = sub.O2_State.to_numpy()
    edges = [0] + list(np.where(st[1:] != st[:-1])[0] + 1) + [len(st)]
    for a in ax:
        for i in range(len(edges) - 1):
            lo, hi = edges[i], edges[i + 1] - 1
            a.axvspan(t[lo], t[hi], color=cmap.get(st[lo], 'gray'), alpha=0.12)
    ax[0].set_title(crop)
    fig.tight_layout(); plt.show()

widgets.interact(show, crop=widgets.Dropdown(options=crops, description='crop'));

### 6. Save tidy summaries (for R / sharing)

In [ ]:
outdir = Path('analysis'); outdir.mkdir(exist_ok=True)
summary.to_csv(outdir / 'per_state_summary.csv', index=False)
tta.to_csv(outdir / 'transition_triggered_forward_velocity.csv', index=False)
print('wrote', list(outdir.glob('*.csv')))